In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd
import os
import kagglehub


In [3]:
# Download latest version
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
files = os.listdir(path)
data = pd.read_csv(os.path.join(path, 'spam.csv'), encoding='latin1')

In [4]:
print(data.shape)
print(data.head(3))
#print column names:
print(data.columns.tolist())

(5572, 5)
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


In [5]:
#clean dataset:
#keep only first two columns: v1 -> spam/ham and v2 -> text message
data = data[['v1', 'v2']].copy()
data.columns = ['label', 'message']
print(data.head(3))

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...


In [6]:
#check for missing data:
missing = data.isnull().sum()
print(missing)

label      0
message    0
dtype: int64


In [7]:
#removing rows with missing values
data = data.dropna()
data = data.dropna().reset_index(drop=True) #reset index
print(data.shape)

(5572, 2)


In [8]:
print(data['label'].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [13]:
#turning labels into binary:
data['label_num'] = data['label'].map({'ham':0, 'spam':1})
data.sample(5)

,label,message,label_num
452,ham,K:)k:)what are detail you want to transfer?acc...,0
122,spam,Todays Voda numbers ending 7548 are selected t...,1
871,ham,Its going good...no problem..but still need li...,0
4897,ham,Oh for fuck's sake she's in like tallahassee,0
1997,ham,\YEH I AM DEF UP4 SOMETHING SAT,0


In [ ]:
import string, re, nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.downnload('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
#preprocessing text:
#tools:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
#converting text meassage to lowercase:
data['clean_message'] = data['message'].str.lower()
#tokenizing
data['clean_message'] = data['clean_message'].apply(word_tokenize)
#removing special characters:
data['clean_message'] = data['clean_message'].apply(lambda x: [re.sub(r'[^a-zA-Z0-9\s]', '', word) for word in x])
#removing stop words and punctuation:
data['clean_message'] = data['clean_message'].apply(lambda x: [word for word in x if word not in stop_words and word not in string.punctuation])
#reduce words to base form:
data['clean_message'] = data['clean_message'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

# Convert the preprocessed text back to string
data['clean_message'] = data['clean_message'].apply(lambda x: ' '.join(x))

# Display the preprocessed data
print(data[['message', 'clean_message']].head())

                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry 2 wkly comp win fa cup final tkts 2...  
3                u dun say early hor u c already say  
4             nah nt think go usf life around though  


In [25]:
print(data.sample(5))


     label                                            message  label_num  \
5460  spam  December only! Had your mobile 11mths+? You ar...          1   
171    ham  Hmmm.. Thk sure got time to hop ard... Ya, can...          0   
4618   ham                 Sorry, I'll call later In meeting.          0   
3405   ham  \HEY DAS COOL... IKNOW ALL 2 WELLDA PERIL OF S...          0   
348   spam  Fancy a shag? I do.Interested? sextextuk.com t...          1   

                                          clean_message  
5460  december mobile 11mths entitled update latest ...  
171   hmmm thk sure got time hop ard ya go 4 free ab...  
4618                           sorry call later meeting  
3405  hey da cool iknow 2 wellda peril studentfinanc...  
348   fancy shag dointerested sextextukcom txt xxuk ...  
